# Double Descent

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt
import numpy as np

## Data Preparation

In [ ]:
def twospirals(n_points, noise=.5, random_state=920):
    """
     Returns the two spirals dataset.
    """
    n = np.sqrt(np.random.rand(n_points,1)) * 600 * (2*np.pi)/360
    d1x = -1.5*np.cos(n)*n + np.random.randn(n_points,1) * noise
    d1y =  1.5*np.sin(n)*n + np.random.randn(n_points,1) * noise
    return (np.vstack((np.hstack((d1x,d1y)),np.hstack((-d1x,-d1y)))),
            np.hstack((np.zeros(n_points),np.ones(n_points))))

X, y = twospirals(500, noise=1.3)

In [ ]:
import torch
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)

X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)
X_test_tensor = X_test_tensor.to(device)
y_test_tensor = y_test_tensor.to(device)

print(device)

## Model

### Definition

In [ ]:
import torch.nn as nn
import torch.optim as optim

class NN(nn.Module):
    def __init__(self, num_hidden_layers=5, neurons_per_layer=20, input_size=2, output_size=1, hidden_sizes=None, activation=None):
        super(NN, self).__init__()
        if hidden_sizes is None:
            hidden_sizes = [neurons_per_layer] * num_hidden_layers
        if activation is None:
            activation = nn.ReLU()

        layers = []
        in_features = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h, bias=True))
            layers.append(activation)
            in_features = h
        layers.append(nn.Linear(in_features, output_size, bias=True))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# Vary width over a large range to cross the interpolation threshold.
# Keep depth fixed so we isolate width as the complexity axis.
depth = 2
widths = list(range(1, 21)) + [25, 32, 40, 50, 64, 80, 100, 128, 200, 300, 512]

### Training

In [ ]:
train_errors = []
test_errors  = []
num_params_list = []

for width in widths:
    model = NN(num_hidden_layers=depth, neurons_per_layer=width, input_size=2, output_size=1, activation=nn.ReLU()).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Train
    num_epochs = 10000
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_tensor), y_train_tensor)
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        train_preds = (model(X_train_tensor) > 0).float()
        train_error = (train_preds != y_train_tensor).float().mean().item()
        test_preds = (model(X_test_tensor) > 0).float()
        test_error = (test_preds != y_test_tensor).float().mean().item()
    
    n_params = sum(p.numel() for p in model.parameters())
    train_errors.append(train_error)
    test_errors.append(test_error)
    num_params_list.append(n_params)

    print(f"Width {width:>3d}:  Train Error: {train_error:.4f},  Test Error: {test_error:.4f},  Params: {n_params}")

## Visualization

In [ ]:
# Without label flipping
plt.figure(figsize=(10, 5))
plt.plot(num_params_list, train_errors, marker='o', label="Train Error")
plt.plot(num_params_list, test_errors,  marker='o', label="Test Error")
plt.xscale('log')
plt.xlabel("Number of Parameters (log scale)")
plt.ylabel("Classification Error (0-1)")
plt.title(f"Double Descent: Depth={depth}, Width varied")

# Mark the interpolation threshold (≈ number of training samples)
n_train = len(X_train)
plt.axvline(x=n_train, color='gray', linestyle='--', alpha=0.7,
            label=f'Interpolation threshold (~{n_train} params)')

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Decision Boundary